### WPROWADZENIE DO RNN
#### Zadanie: Naucz sieć przewidywać kolejną literę w zdaniu!

In [2]:
import torch
import torch.nn as nn
import numpy as np

#### CZĘŚĆ 1: PPRZYGOTOWANIE DANYCH (nie modyfikuj)

In [5]:
tekst = "student rozumie sieci rekurencyjne "
alfabet = sorted(list(set(tekst)))
rozmiar_alfabetu = len(alfabet)

In [7]:
alfabet

[' ',
 'c',
 'd',
 'e',
 'i',
 'j',
 'k',
 'm',
 'n',
 'o',
 'r',
 's',
 't',
 'u',
 'y',
 'z']

In [9]:
rozmiar_alfabetu

16

In [13]:
# Słowniki do tłumaczenia: znak <-> liczba
char_to_idx = {ch: i for i, ch in enumerate(alfabet)}
idx_to_char = {i: ch for i, ch in enumerate(alfabet)}

In [15]:
char_to_idx

{' ': 0,
 'c': 1,
 'd': 2,
 'e': 3,
 'i': 4,
 'j': 5,
 'k': 6,
 'm': 7,
 'n': 8,
 'o': 9,
 'r': 10,
 's': 11,
 't': 12,
 'u': 13,
 'y': 14,
 'z': 15}

In [17]:
idx_to_char

{0: ' ',
 1: 'c',
 2: 'd',
 3: 'e',
 4: 'i',
 5: 'j',
 6: 'k',
 7: 'm',
 8: 'n',
 9: 'o',
 10: 'r',
 11: 's',
 12: 't',
 13: 'u',
 14: 'y',
 15: 'z'}

In [19]:
# Parametry sekwencji
seq_length = 5 # Sieć czyta 5 liter i zgaduje 6-tą
X_dane = []
y_dane = []

In [21]:
# Cięcie tekstu na okna przesuwne
for i in range(len(tekst) - seq_length):
    sekwencja = tekst[i:i+seq_length]
    nastepny_znak = tekst[i+seq_length]
    
    # Kodowanie One-Hot dla wejścia X
    x_one_hot = np.zeros((seq_length, rozmiar_alfabetu))
    for j, znak in enumerate(sekwencja):
        x_one_hot[j, char_to_idx[znak]] = 1
    
    X_dane.append(x_one_hot)
    y_dane.append(char_to_idx[nastepny_znak])

In [23]:
# Konwersja na Tensory PyTorch
# X ma wymiary: [Wielkość_paczki, Długość_sekwencji, Rozmiar_wejścia]
X = torch.tensor(np.array(X_dane), dtype=torch.float32)
Y = torch.tensor(np.array(y_dane), dtype=torch.long)

print(f"Kształt tensora wejściowego X: {X.shape}")

Kształt tensora wejściowego X: torch.Size([30, 5, 16])


#### CZĘŚĆ 2: TWOJE ZADANIE (uzupełnij kod)

In [26]:
class MojaSiecRNN(nn.Module):
    def __init__(self, rozmiar_wejscia, rozmiar_pamieci, liczba_klas):
        super(MojaSiecRNN, self).__init__()
        
        self.rozmiar_pamieci = rozmiar_pamieci
        
        # TODO 1: Zainicjuj wbudowany moduł nn.RNN. 
        # Pamiętaj o parametrach: input_size oraz hidden_size.
        # UWAGA: Nasz tensor X ma wielkość paczki na pierwszym miejscu. 
        # Jakiej flagi musisz użyć, aby PyTorch to zrozumiał?
        self.rnn = None 
        
        # TODO 2: Zainicjuj warstwę liniową (nn.Linear), która zamieni wynik z RNN 
        # (czyli rozmiar_pamieci) na ostateczne prawdopodobieństwa liter (liczba_klas).
        self.fc = None

    def forward(self, x):
        # TODO 3: Stwórz "czystą kartę" (stan początkowy h_0) składający się z samych zer.
        # Użyj torch.zeros(...). 
        # Pamiętaj, że wymiar h_0 to ZAWSZE: [1, wielkosc_paczki, pojemnosc_pamieci]
        # Wskazówka: Wielkość paczki wyciągniesz z wejścia za pomocą x.size(0).
        h0 = None
        
        # TODO 4: Przepuść dane 'x' oraz stan 'h0' przez zdefiniowany wcześniej silnik rnn.
        # Pamiętaj, że nn.RNN zwraca dwie wartości! Zapisz je do zmiennych 'out' i 'hidden'.
        out, hidden = None, None
        
        # TODO 5: Architektura Many-to-One. Nasza sieć ma wypluć tylko JEDNĄ odpowiedź 
        # (kolejną literę) po przeczytaniu całej 5-znakowej sekwencji. 
        # Nadpisz zmienną 'out', wycinając z niej tylko ostatni krok czasowy.
        out = None 
        
        # TODO 6: Przepuść uzyskany wynik przez warstwę liniową.
        out = None
        
        return out

#### CZĘŚĆ 3: PĘTLA UCZĄCA I TESTOWANIE (nie modyfikuj)

In [29]:
# Inicjalizacja modelu (pamięć = 32 ukryte neurony)
model = MojaSiecRNN(rozmiar_wejscia=rozmiar_alfabetu, rozmiar_pamieci=32, liczba_klas=rozmiar_alfabetu)

In [33]:
# nn.CrossEntropyLoss zawiera już w sobie LogSoftmax! 
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

print("\nRozpoczynam trening...")
epoki = 200

try:
    for epoka in range(epoki):
        optimizer.zero_grad()
        
        wyjscie = model(X)
        blad = criterion(wyjscie, Y)
        
        blad.backward()
        optimizer.step()
        
        if (epoka+1) % 50 == 0:
            print(f"Epoka: {epoka+1}/{epoki}, Błąd (Loss): {blad.item():.4f}")

    print("\nTRENING ZAKOŃCZONY SUKCESEM!")
    print("Sprawdźmy, czy sieć umie kontynuować zdanie na podstawie pierwszych 5 liter ('stude').")
    
    # Testowanie (Generowanie tekstu)
    wygenerowany_tekst = "stude"
    
    model.eval()
    for i in range(30): # Wygeneruj 30 kolejnych znaków
        # Przygotuj ostatnie 5 liter do formatu One-Hot
        ost_sekwencja = wygenerowany_tekst[-5:]
        x_test = np.zeros((1, 5, rozmiar_alfabetu))
        for j, znak in enumerate(ost_sekwencja):
            x_test[0, j, char_to_idx[znak]] = 1
            
        x_tensor = torch.tensor(x_test, dtype=torch.float32)
        
        # Zapytaj sieć
        with torch.no_grad():
            predykcja = model(x_tensor)
            
        # Wybierz literę z największym prawdopodobieństwem
        idx_najlepszej = torch.argmax(predykcja).item()
        wygenerowany_tekst += idx_to_char[idx_najlepszej]

    print(f"\nWynik sieci: '{wygenerowany_tekst}'")

except TypeError as e:
    print(f"\n[BŁĄD] Kod nie jest jeszcze gotowy. Uzupełnij wszystkie sekcje TODO! Szczegóły: {e}")
except RuntimeError as e:
    print(f"\n[BŁĄD WYMIARÓW] Coś poszło nie tak z tensorami. Sprawdź TODO 1, 3 lub 5. Szczegóły: {e}")

ValueError: optimizer got an empty parameter list